# LoRA-NLLB End-to-End Sichuanese -> English
#
Your setting:
- You are responsible for one model: LoRA-NLLB end-to-end `SC -> EN`.
- `train_9800_zh.csv` has no English column, only:
  - `text`: Sichuanese source
  - `mandarin`: Mandarin pivot
- `golden_manually_200_en.csv` has the 200-row test set with gold/reference English.
#
Pipeline:
1. Use vanilla NLLB to translate train Mandarin -> pseudo English.
2. Fine-tune LoRA-NLLB on `text -> pseudo_english_nllb`.
3. Run LoRA-NLLB on test `text -> pred_english_lora_nllb`.
4. Evaluate against test gold English with BLEU and chrF.
#
Kaggle input:
- Upload `train_9800_zh.csv` and `golden_manually_200_en.csv` to one Kaggle Dataset.
- Attach that Dataset to this notebook.
- Enable GPU and Internet.
#
Outputs:
- `/kaggle/working/train_9800_zh_pseudo_en.csv`
- `/kaggle/working/lora_nllb_sc_en_predictions.csv`
- `/kaggle/working/lora_nllb_sc_en_metrics.csv`
- `/kaggle/working/lora_nllb_sc_en_cases.csv`
- `/kaggle/working/nllb_lora_sc_en_adapter/`


## 0. Install Packages

Run once per Kaggle session. **Success** = you see `Installed stack:` with version numbers below.

| Message | Action |
|---------|--------|
| `dependency conflicts` for bigframes / s3fs / gcsfs / tpot | Ignore — unrelated to NLLB training |
| `Unable to register cuFFT/cuDNN/cuBLAS factory` | Ignore — TensorFlow + PyTorch on same image |
| `ImportError` in the next cell | Restart session → run §0 → imports again |

After a successful install, set `INSTALL_PACKAGES = False` on re-runs to skip pip.


In [1]:
!pip uninstall -y torchao


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [2]:
import os
import re
import sys
import glob
import random
import subprocess
from pathlib import Path

try:
    from IPython.display import display
except ImportError:
    display = print

# Set False after §0 succeeds once in this session (avoids re-triggering pip warnings).
INSTALL_PACKAGES = True

KAGGLE_ML_STACK = [
    "transformers==4.46.3",
    "accelerate==1.2.1",
    "peft==0.14.0",
    "datasets==3.2.0",
    "sacrebleu==2.4.3",
    "sentencepiece>=0.2.0",
    "protobuf>=3.20,<6",
]


def _pip_install(packages):
    return subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade-strategy",
            "only-if-needed",
            *packages,
        ],
        capture_output=True,
        text=True,
    )


def _summarize_pip_conflicts(pip_log):
    """Show only package lines; skip pip's generic ERROR header."""
    pkg_lines = [
        ln.strip()
        for ln in pip_log.splitlines()
        if " requires " in ln.lower() or " but you have " in ln.lower()
    ]
    if not pkg_lines:
        return
    print("Optional Kaggle-only pip notices (safe to ignore for this notebook):")
    for ln in pkg_lines[:6]:
        print("  -", ln)
    if len(pkg_lines) > 6:
        print(f"  ... and {len(pkg_lines) - 6} more")


def kaggle_install_ml_stack():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"],
        check=False,
    )
    result = _pip_install(KAGGLE_ML_STACK)
    pip_log = (result.stdout or "") + (result.stderr or "")
    if result.returncode != 0:
        fallback = [
            "transformers>=4.45,<5",
            "accelerate>=1.1,<2",
            "peft>=0.13,<0.16",
            "datasets>=3.0,<4",
            "sacrebleu>=2.4",
            "sentencepiece>=0.2.0",
            "protobuf>=3.20,<6",
        ]
        print("Pinned install failed; retrying with compatible ranges...")
        result = _pip_install(fallback)
        pip_log = (result.stdout or "") + (result.stderr or "")
    if result.returncode != 0:
        raise RuntimeError(
            "pip install failed.\n"
            f"stdout:\n{result.stdout}\n"
            f"stderr:\n{result.stderr}"
        )

    import accelerate
    import datasets
    import peft
    import transformers

    print("SUCCESS — ML stack ready for NLLB + LoRA:")
    print("  transformers:", transformers.__version__)
    print("  accelerate:  ", accelerate.__version__)
    print("  peft:        ", peft.__version__)
    print("  datasets:    ", datasets.__version__)

    if "dependency conflicts" in pip_log.lower():
        _summarize_pip_conflicts(pip_log)
        print("Continue to the import cell (§0 next code cell). CUDA factory messages are normal on Kaggle.")


if INSTALL_PACKAGES:
    kaggle_install_ml_stack()
else:
    print("Skipping pip (INSTALL_PACKAGES=False). Using packages already in this session.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.9 MB/s eta 0:00:00


2026-05-20 10:50:36.071700: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779274236.330130      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779274236.403191      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779274237.007409      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779274237.007453      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779274237.007455      57 computation_placer.cc:177] computation placer alr

SUCCESS — ML stack ready for NLLB + LoRA:
  transformers: 4.46.3
  accelerate:   1.2.1
  peft:         0.14.0
  datasets:     3.2.0
Optional Kaggle-only pip notices (safe to ignore for this notebook):
  - bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
  - s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.9.0 which is incompatible.
  - tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
  - gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
Continue to the import cell (§0 next code cell). CUDA factory messages are normal on Kaggle.


## 1. Config


In [3]:
import numpy as np
import pandas as pd
import torch
import sacrebleu
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)


In [4]:
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

INPUT_ROOT = "/kaggle/input"
WORK_DIR = Path("/kaggle/working")
WORK_DIR.mkdir(parents=True, exist_ok=True)



PSEUDO_CSV_NAME = "train_9800_zh_pseudo_en.csv"
# Set explicit path if auto-search fails, e.g. "/kaggle/input/.../train_9800_zh_pseudo_en.csv"
PSEUDO_CSV_PATH = None


def resolve_pseudo_csv_path():
    """Find pseudo-label CSV under WORK_DIR, cwd, or Kaggle input (no find_csv dependency)."""
    tried = []
    if PSEUDO_CSV_PATH is not None:
        tried.append(Path(PSEUDO_CSV_PATH))
    tried.extend([
        WORK_DIR / PSEUDO_CSV_NAME,
        Path(PSEUDO_CSV_NAME),
    ])
    tried.extend(Path(p) for p in glob.glob(str(WORK_DIR / "**" / PSEUDO_CSV_NAME), recursive=True))
    tried.extend(Path(p) for p in glob.glob(os.path.join(INPUT_ROOT, "**", PSEUDO_CSV_NAME), recursive=True))
    tried.extend(Path(p) for p in glob.glob(os.path.join(INPUT_ROOT, "**", "*pseudo*en*.csv"), recursive=True))

    seen = set()
    for p in tried:
        ps = str(p.resolve()) if p.exists() else str(p)
        if ps in seen:
            continue
        seen.add(ps)
        if p.exists():
            return p
    return WORK_DIR / PSEUDO_CSV_NAME


def load_pseudo_labels_into_train_df(pseudo_path):
    pseudo_df = read_csv_flexible(pseudo_path)
    if PSEUDO_EN_COL not in pseudo_df.columns:
        raise ValueError(f"{pseudo_path} missing column {PSEUDO_EN_COL}")

    key_col = "text" if "text" in pseudo_df.columns else TRAIN_SC_COL
    pseudo_df = pseudo_df[[key_col, PSEUDO_EN_COL]].copy()
    pseudo_df[key_col] = pseudo_df[key_col].astype(str).str.strip()
    pseudo_df = pseudo_df.rename(columns={key_col: TRAIN_SC_COL})
    pseudo_df = pseudo_df.dropna(subset=[TRAIN_SC_COL, PSEUDO_EN_COL])
    pseudo_df = pseudo_df.drop_duplicates(subset=[TRAIN_SC_COL], keep="first")

    # map keeps train_df length (merge can explode on duplicate text keys)
    lookup = dict(zip(pseudo_df[TRAIN_SC_COL], pseudo_df[PSEUDO_EN_COL]))
    train_df[PSEUDO_EN_COL] = train_df[TRAIN_SC_COL].astype(str).str.strip().map(lookup)

    n_miss = int(train_df[PSEUDO_EN_COL].isna().sum())
    if n_miss > len(train_df) * 0.05 and len(pseudo_df) == len(train_df):
        train_df[PSEUDO_EN_COL] = pseudo_df[PSEUDO_EN_COL].values
        n_miss = int(train_df[PSEUDO_EN_COL].isna().sum())
        print("Warning: text key mismatch; aligned pseudo labels by row order.")
    elif n_miss > len(train_df) * 0.05:
        raise ValueError(
            f"{n_miss}/{len(train_df)} rows missing pseudo EN. "
            "Check train_9800_zh vs pseudo CSV text column alignment."
        )
    return n_miss


TRAIN_CSV = None
TEST_CSV = None

SC_COL = "text"
ZH_COL = "mandarin"
GOLD_EN_COL = "english"

PSEUDO_EN_COL = "pseudo_english_nllb"
PRED_EN_COL = "pred_english_lora_nllb"

NLLB_MODEL_NAME = "facebook/nllb-200-distilled-600M"

NLLB_SRC_LANG = "zho_Hans"
NLLB_TGT_LANG = "eng_Latn"

MAX_SOURCE_LEN = 96
MAX_TARGET_LEN = 128
MAX_NEW_TOKENS_EN = 128

NLLB_BATCH_SIZE = 16
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
NUM_TRAIN_EPOCHS = 2
LEARNING_RATE = 5e-5
DEV_SIZE = 500

TRAIN_LIMIT = None
TEST_LIMIT = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TRAIN_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
INFER_DTYPE = torch.float32
# Alias for §3 pseudo cell (uses torch_dtype=DTYPE); do not remove.
DTYPE = TRAIN_DTYPE


def batch_iter(items, batch_size):
    for i in range(0, len(items), batch_size):
        yield items[i : i + batch_size]


def get_nllb_tgt_lang_id(tokenizer):
    if hasattr(tokenizer, "lang_code_to_id") and NLLB_TGT_LANG in tokenizer.lang_code_to_id:
        return tokenizer.lang_code_to_id[NLLB_TGT_LANG]
    return tokenizer.convert_tokens_to_ids(NLLB_TGT_LANG)


def unwrap_nllb_core(model):
    if hasattr(model, "get_base_model"):
        return model.get_base_model()
    if hasattr(model, "base_model") and hasattr(model.base_model, "model"):
        return model.base_model.model
    return model




def prepare_nllb_lora_for_training(model, use_gradient_checkpointing=True):
    """PEFT + checkpointing: enable grads on embeddings; use_reentrant=False; no Trainer double-enable."""
    core = unwrap_nllb_core(model)
    core.config.use_cache = False
    if hasattr(core, "generation_config") and core.generation_config is not None:
        core.generation_config.use_cache = False

    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    shared = getattr(core, "shared", None)
    if shared is None and hasattr(core, "model"):
        shared = getattr(core.model, "shared", None)
    if shared is not None:
        def _emb_out_requires_grad(_module, _inp, out):
            out.requires_grad_(True)
        if not getattr(shared, "_nllb_lora_grad_hook", False):
            shared.register_forward_hook(_emb_out_requires_grad)
            shared._nllb_lora_grad_hook = True

    if use_gradient_checkpointing:
        gc_kwargs = {"use_reentrant": False}
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gc_kwargs)
        except TypeError:
            model.gradient_checkpointing_enable()
    else:
        if hasattr(model, "gradient_checkpointing_disable"):
            model.gradient_checkpointing_disable()

    model.train()
    return model

def configure_nllb_train_config(model, tokenizer):
    """Training only: decoder_start == eng_Latn (NOT 2). Never use decoder_start_token_id=2 for NLLB."""
    tid = get_nllb_tgt_lang_id(tokenizer)
    core = unwrap_nllb_core(model)
    core.config.forced_bos_token_id = tid
    core.config.decoder_start_token_id = tid
    if hasattr(core, "generation_config") and core.generation_config is not None:
        core.generation_config.forced_bos_token_id = tid
        core.generation_config.decoder_start_token_id = tid
    if hasattr(model, "config"):
        model.config.forced_bos_token_id = tid
        model.config.decoder_start_token_id = tid
    return tid


def mask_nllb_label_ids(label_ids, tokenizer):
    lang_id = get_nllb_tgt_lang_id(tokenizer)
    return [[-100 if tok_id == lang_id else tok_id for tok_id in seq] for seq in label_ids]


def looks_bad_pseudo_en(text, min_words=5, max_unique_ratio=0.45):
    words = str(text).strip().split()
    if len(words) < min_words:
        return True
    return (len(set(w.lower() for w in words)) / len(words)) < max_unique_ratio


@torch.inference_mode()
def nllb_translate_lora_fp32(model, tokenizer, texts, batch_size=16, max_new_tokens=MAX_NEW_TOKENS_EN):
    """Predict: fp32 on full PeftModel (Do NOT generate only on base — skips LoRA). Same kwargs as §3."""
    model.eval()
    tid = get_nllb_tgt_lang_id(tokenizer)
    model = model.float().to(DEVICE)
    outputs = []
    texts = [str(x).strip() for x in texts]
    for batch in batch_iter(texts, batch_size):
        tokenizer.src_lang = NLLB_SRC_LANG
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        ).to(DEVICE)
        generated = model.generate(
            **inputs,
            forced_bos_token_id=tid,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            length_penalty=1.0,
            early_stopping=True,
            no_repeat_ngram_size=3,
        )
        outputs.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return [x.strip() for x in outputs]


print("Device:", DEVICE)
print("NLLB model:", NLLB_MODEL_NAME)


Device: cuda
NLLB model: facebook/nllb-200-distilled-600M


## 2. Load CSVs


In [5]:
def find_csv(preferred_names, required=True):
    csvs = sorted(glob.glob(os.path.join(INPUT_ROOT, "**", "*.csv"), recursive=True))
    for preferred in preferred_names:
        matches = [p for p in csvs if preferred.lower() in os.path.basename(p).lower()]
        if matches:
            return sorted(matches, key=len)[0]
    if required:
        raise FileNotFoundError(
            f"Could not find any of {preferred_names} under {INPUT_ROOT}. "
            f"Found CSVs: {csvs[:30]}"
        )
    return None


def read_csv_flexible(path):
    for enc in ["utf-8-sig", "utf-8", "gb18030"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path)


if TRAIN_CSV is None:
    TRAIN_CSV = find_csv(["train_9800_zh", "train_9800_mandarin", "train_9800"])
if TEST_CSV is None:
    TEST_CSV = find_csv(
        ["golden_manually_200_en", "test_200_en", "tr_test_200_en", "test_200"]
    )

train_df = read_csv_flexible(TRAIN_CSV)
test_df = read_csv_flexible(TEST_CSV)

print("TRAIN_CSV:", TRAIN_CSV)
print("TEST_CSV:", TEST_CSV)
print("Train:", train_df.shape, list(train_df.columns))
print("Test: ", test_df.shape, list(test_df.columns))
display(train_df.head(3))
display(test_df.head(3))


TRAIN_CSV: /kaggle/input/datasets/yhclywl/sichuanese-english-translation/train_9800_zh.csv
TEST_CSV: /kaggle/input/datasets/yhclywl/sichuanese-english-translation/golden_manually_200_en.csv
Train: (9800, 6) ['source_file', 'utt_or_id', 'filename', 'text', 'matched_terms', 'mandarin']
Test:  (200, 14) ['source_file', 'utt_or_id', 'filename', 'text', 'matched_terms', 'mandarin', 'english', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13']


,source_file,utt_or_id,filename,text,matched_terms,mandarin
0,wsc_train_audio_text.tsv.gz,cq00015015_65860_81310,cq00015015_65860_81310.wav,老房子家具家电齐全直接就可以拎包入住这个位置这个户型这个装修你们喜欢吗喜欢的话记到联系我哟上...,你们 这个 你 我,老房子家具家电齐全，可以直接拎包入住。这个位置、这个户型、这个装修，你们喜欢吗？如果喜欢的话...
1,wsc_train_audio_text.tsv.gz,cq00009746_159450_171270,cq00009746_159450_171270.wav,反正他车在那里放着也是放着租不出去也损失了那么我粉丝比如说正常正常a六该住六百一天的该住五百...,粉丝 我 他,反正他的车放在那里也是放着，租不出去也会有损失。比如我的粉丝，比如说正常的A6应该住600一...
2,wsc_train_audio_text.tsv.gz,cq00001785_7080_20010,cq00001785_7080_20010.wav,客户直接问一下那是你们幺儿吗你说我哪儿像那长得像那个同事他们妈咹我到底比他老好多你们说我看起...,他们 同事 那个 幺儿 你们 你 我 他 妈,客户直接问一下，那是你们的孩子吗？我说我哪儿像呢，长得不像那个同事他们妈妈呢？我到底比他大好...


,source_file,utt_or_id,filename,text,matched_terms,mandarin,english,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,wsc_train_audio_text.tsv.gz,704219202372_Al7Ir_22_3640,704219202372_Al7Ir_22_3640.wav,我是捡镜子的命法莫法生了这个命哦命哦,这个 我,我这是捡镜子的命，没办法，生来就是这个命啊。,I guess this is my fate—to pick up mirrors. Th...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,wsc_train_audio_text.tsv.gz,cq0034024_86130_88550,cq0034024_86130_88550.wav,谢谢谢谢李婆婆,婆婆,谢谢，谢谢李婆婆。,"Thank you, Grandma Li.",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,wsc_train_audio_text.tsv.gz,cq0034279_0_9560,cq0034279_0_9560.wav,我有几个朋友都强烈推荐他们家的小面说海椒香得很点一碗干的大款来搞哈调料就是这几种也不复杂没得...,他们 朋友 我,我有几个朋友都强烈推荐他们家的小面，说辣椒特别香。点一碗干拌的大碗面来尝一下，调料就是这几种...,Several of my friends strongly recommended the...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
def norm_name(name):
    return str(name).strip().lower().replace(" ", "_").replace("-", "_")


def guess_col(df, candidates, role, required=True):
    if role == "sc" and SC_COL is not None and SC_COL in df.columns:
        return SC_COL
    if role == "zh" and ZH_COL is not None and ZH_COL in df.columns:
        return ZH_COL
    if role == "en" and GOLD_EN_COL is not None and GOLD_EN_COL in df.columns:
        return GOLD_EN_COL

    normalized = {norm_name(c): c for c in df.columns}
    for cand in candidates:
        cand_norm = norm_name(cand)
        for n, original in normalized.items():
            if cand_norm == n or cand_norm in n:
                return original
    if required:
        raise ValueError(
            f"Could not infer {role} column. Columns: {list(df.columns)}. "
            "Set SC_COL / ZH_COL / GOLD_EN_COL manually in the config cell."
        )
    return None


SC_CANDIDATES = [
    "text",
    "sc",
    "sichuanese",
    "sichuan",
    "dialect",
    "source",
    "src",
    "四川话",
    "川话",
    "方言",
    "原文",
]
ZH_CANDIDATES = [
    "mandarin",
    "zh",
    "chinese",
    "pivot",
    "normalization",
    "普通话",
    "中文",
    "中介",
]
EN_CANDIDATES = [
    "gold_english",
    "reference_english",
    "ref_english",
    "gold_en",
    "english",
    "en",
    "target",
    "translation",
    "英文",
    "英语",
]

TRAIN_SC_COL = guess_col(train_df, SC_CANDIDATES, "sc")
TRAIN_ZH_COL = guess_col(train_df, ZH_CANDIDATES, "zh")
TEST_SC_COL = guess_col(test_df, SC_CANDIDATES, "sc")
TEST_EN_COL = guess_col(test_df, EN_CANDIDATES, "en")

print("Detected columns:")
print("  train SC:", TRAIN_SC_COL)
print("  train ZH:", TRAIN_ZH_COL)
print("  test SC: ", TEST_SC_COL)
print("  test EN: ", TEST_EN_COL)


Detected columns:
  train SC: text
  train ZH: mandarin
  test SC:  text
  test EN:  english


## 3. Pseudo English labels

**Default: load** `train_9800_zh_pseudo_en.csv` — **does not** run NLLB.

Put file in `/kaggle/working/` or attach to input dataset. Or set `PSEUDO_CSV_PATH = "/kaggle/input/.../train_9800_zh_pseudo_en.csv"` in Config.

Only set `FORCE_REGENERATE_PSEUDO = True` to regenerate.


In [7]:
def clean_text_series(s):
    return s.astype(str).str.strip().replace({"": np.nan, "nan": np.nan, "None": np.nan})


train_df[TRAIN_SC_COL] = clean_text_series(train_df[TRAIN_SC_COL])
train_df[TRAIN_ZH_COL] = clean_text_series(train_df[TRAIN_ZH_COL])
train_df = train_df.dropna(subset=[TRAIN_SC_COL, TRAIN_ZH_COL]).reset_index(drop=True)
if TRAIN_LIMIT is not None:
    train_df = train_df.head(TRAIN_LIMIT).copy()

test_df[TEST_SC_COL] = clean_text_series(test_df[TEST_SC_COL])
test_df[TEST_EN_COL] = clean_text_series(test_df[TEST_EN_COL])
test_df = test_df.dropna(subset=[TEST_SC_COL, TEST_EN_COL]).reset_index(drop=True)
if TEST_LIMIT is not None:
    test_df = test_df.head(TEST_LIMIT).copy()

print("Clean train:", train_df.shape)
print("Clean test: ", test_df.shape)


Clean train: (9800, 6)
Clean test:  (200, 14)


In [8]:
# §3: load pseudo EN from CSV (default). Generate only if FORCE_REGENERATE_PSEUDO = True.
FORCE_REGENERATE_PSEUDO = False
pseudo_path = resolve_pseudo_csv_path()
print("Pseudo CSV path:", pseudo_path, "| exists:", pseudo_path.exists())

if FORCE_REGENERATE_PSEUDO:
    import torch
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

    print("FORCE_REGENERATE_PSEUDO=True — running NLLB on Mandarin...")
    nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME)
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_MODEL_NAME, torch_dtype=DTYPE).to(DEVICE)
    nllb_model.eval()
    nllb_tokenizer.src_lang = "zho_Hans"
    tgt_lang_id = nllb_tokenizer.convert_tokens_to_ids("eng_Latn")

    @torch.inference_mode()
    def nllb_zh_to_en(texts, batch_size=NLLB_BATCH_SIZE):
        outputs = []
        texts = [str(x).strip() for x in texts]
        for batch in batch_iter(texts, batch_size):
            inputs = nllb_tokenizer(
                batch, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SOURCE_LEN,
            ).to(DEVICE)
            generated = nllb_model.generate(
                **inputs, forced_bos_token_id=tgt_lang_id, max_new_tokens=MAX_NEW_TOKENS_EN,
                num_beams=4, length_penalty=1.0, early_stopping=True, no_repeat_ngram_size=3,
            )
            outputs.extend(nllb_tokenizer.batch_decode(generated, skip_special_tokens=True))
            print(f"Generated pseudo EN: {len(outputs)}/{len(texts)}", end="\r")
        print()
        return [x.strip() for x in outputs]

    train_df[PSEUDO_EN_COL] = nllb_zh_to_en(train_df[TRAIN_ZH_COL].tolist())
    pseudo_save = train_df[[TRAIN_SC_COL, TRAIN_ZH_COL, PSEUDO_EN_COL]].rename(
        columns={TRAIN_SC_COL: "text", TRAIN_ZH_COL: "mandarin"}
    )
    save_path = WORK_DIR / PSEUDO_CSV_NAME
    pseudo_save.to_csv(save_path, index=False, encoding="utf-8-sig")
    print("Saved:", save_path)
    del nllb_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    if not pseudo_path.exists():
        raise FileNotFoundError(
            f"Pseudo CSV not found. Copy {PSEUDO_CSV_NAME} to /kaggle/working/ "
            f"or set PSEUDO_CSV_PATH in Config. Will NOT auto-regenerate "
            f"(set FORCE_REGENERATE_PSEUDO=True only if you intend to rebuild)."
        )
    n_miss = load_pseudo_labels_into_train_df(pseudo_path)
    print(f"Loaded pseudo EN from: {pseudo_path}")
    print(f"  rows={len(train_df)}, missing pseudo={n_miss}")

display(train_df[[TRAIN_ZH_COL, PSEUDO_EN_COL]].head(3))
if looks_bad_pseudo_en(str(train_df[PSEUDO_EN_COL].iloc[0])):
    raise RuntimeError("Pseudo labels look bad — fix §3 before §4.")


Pseudo CSV path: /kaggle/input/datasets/yhclywl/sichuanese-english-translation/train_9800_zh_pseudo_en.csv | exists: True
Loaded pseudo EN from: /kaggle/input/datasets/yhclywl/sichuanese-english-translation/train_9800_zh_pseudo_en.csv
  rows=9800, missing pseudo=0


,mandarin,pseudo_english_nllb
0,老房子家具家电齐全，可以直接拎包入住。这个位置、这个户型、这个装修，你们喜欢吗？如果喜欢的话...,The old house furniture is full of electricity...
1,反正他的车放在那里也是放着，租不出去也会有损失。比如我的粉丝，比如说正常的A6应该住600一...,"Instead of his car being left there, it's also..."
2,客户直接问一下，那是你们的孩子吗？我说我哪儿像呢，长得不像那个同事他们妈妈呢？我到底比他大好...,"If the client asks you directly, is that your ..."


## 4. Fine-tune LoRA-NLLB: Sichuanese -> English

**Must:** `decoder_start_token_id == eng_Latn` (same id as `forced_bos_token_id`). **`decoder_start_token_id = 2` breaks training** → collapsed predict (`t`/junk).


In [9]:
import shutil
from pathlib import Path

for p in ["nllb_lora_sc_en_adapter", "nllb_lora_sc_en_checkpoints"]:
    path = WORK_DIR / p
    if path.exists():
        shutil.rmtree(path)
        print("deleted", path)

if PSEUDO_EN_COL not in train_df.columns or train_df[PSEUDO_EN_COL].isna().mean() > 0.5:
    _pp = resolve_pseudo_csv_path()
    if _pp.exists():
        load_pseudo_labels_into_train_df(_pp)
        print("Merged pseudo from:", _pp)
    else:
        raise FileNotFoundError(f"Missing {PSEUDO_CSV_NAME} — run §3 or set PSEUDO_CSV_PATH.")

train_pairs = train_df[[TRAIN_SC_COL, PSEUDO_EN_COL]].copy()
train_pairs.columns = ["src", "tgt"]
train_pairs["src"] = clean_text_series(train_pairs["src"])
train_pairs["tgt"] = clean_text_series(train_pairs["tgt"])
train_pairs = train_pairs.dropna(subset=["src", "tgt"]).drop_duplicates().reset_index(drop=True)

dev_size = min(DEV_SIZE, max(1, int(0.1 * len(train_pairs))))
shuffled = train_pairs.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
dev_pairs = shuffled.iloc[:dev_size].reset_index(drop=True)
fit_pairs = shuffled.iloc[dev_size:].reset_index(drop=True)

print("Fit pairs:", fit_pairs.shape)
print("Dev pairs:", dev_pairs.shape)


Fit pairs: (9233, 2)
Dev pairs: (500, 2)


In [10]:
from peft import LoraConfig, TaskType, get_peft_model

nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_MODEL_NAME)
# fp32 weights + Trainer fp16 autocast avoids frozen-fp16 + checkpoint grad breaks
nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
    NLLB_MODEL_NAME, torch_dtype=torch.float32
).to(DEVICE)

nllb_tokenizer.src_lang = NLLB_SRC_LANG
nllb_tokenizer.tgt_lang = NLLB_TGT_LANG
tgt_lang_id = configure_nllb_train_config(nllb_model, nllb_tokenizer)
print("eng_Latn id:", tgt_lang_id)
print("decoder_start_token_id (must equal above):", unwrap_nllb_core(nllb_model).config.decoder_start_token_id)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "wi_0", "wi_1", "wo"],
)
nllb_model = get_peft_model(nllb_model, lora_config)
configure_nllb_train_config(nllb_model, nllb_tokenizer)
prepare_nllb_lora_for_training(nllb_model, use_gradient_checkpointing=True)

nllb_model.print_trainable_parameters()
_n_trainable = sum(p.numel() for p in nllb_model.parameters() if p.requires_grad)
print("trainable param count:", _n_trainable)
assert _n_trainable > 0, "LoRA has no trainable parameters"


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

eng_Latn id: 256047
decoder_start_token_id (must equal above): 256047
trainable params: 3,538,944 || all params: 618,612,736 || trainable%: 0.5721
trainable param count: 3538944


In [11]:
fit_ds = Dataset.from_pandas(fit_pairs, preserve_index=False)
dev_ds = Dataset.from_pandas(dev_pairs, preserve_index=False)


def preprocess_nllb(batch):
    nllb_tokenizer.src_lang = NLLB_SRC_LANG
    inputs = nllb_tokenizer(
        batch["src"],
        max_length=MAX_SOURCE_LEN,
        truncation=True,
    )
    nllb_tokenizer.tgt_lang = NLLB_TGT_LANG
    labels = nllb_tokenizer(
        text_target=batch["tgt"],
        max_length=MAX_TARGET_LEN,
        truncation=True,
    )
    inputs["labels"] = mask_nllb_label_ids(labels["input_ids"], nllb_tokenizer)
    return inputs


fit_tok = fit_ds.map(
    preprocess_nllb,
    batched=True,
    remove_columns=fit_ds.column_names,
    load_from_cache_file=False,
)
dev_tok = dev_ds.map(
    preprocess_nllb,
    batched=True,
    remove_columns=dev_ds.column_names,
    load_from_cache_file=False,
)

nm = [x for x in fit_tok[0]["labels"] if x != -100]
print("non-masked label tokens:", len(nm), nm[:10])

data_collator = DataCollatorForSeq2Seq(
    tokenizer=nllb_tokenizer,
    model=nllb_model,
    label_pad_token_id=-100,
)


Map:   0%|          | 0/9233 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

non-masked label tokens: 30 [1603, 196485, 248079, 349, 2601, 248116, 248066, 192197, 3277, 2294]


In [12]:
# 取 fit_tok[0]，把 labels 里非 -100 的 id 组成列表再 decode 看一眼
tos = [t for t in fit_tok[0]["labels"] if t != -100]
print(nllb_tokenizer.decode(tos, skip_special_tokens=True)[:400])

Unfortunately, the ring's certificate has not yet been found, otherwise I would definitely go to the attraction in Tongjiang to play.


In [13]:
import gc

if torch.cuda.is_available():
    gc.collect()
    torch.cuda.empty_cache()

USE_FP16_TRAIN = torch.cuda.is_available()
prepare_nllb_lora_for_training(nllb_model, use_gradient_checkpointing=True)

training_args_kwargs = dict(
    output_dir=str(WORK_DIR / "nllb_lora_sc_en_checkpoints"),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    max_grad_norm=1.0,
    warmup_ratio=0.03,
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=False,
    fp16=USE_FP16_TRAIN,
    bf16=False,
    gradient_checkpointing=False,
    dataloader_pin_memory=False,
    report_to="none",
    save_total_limit=2,
    load_best_model_at_end=False,
)

try:
    training_args = Seq2SeqTrainingArguments(**training_args_kwargs, eval_strategy="no")
except TypeError:
    training_args = Seq2SeqTrainingArguments(**training_args_kwargs, evaluation_strategy="no")

print("USE_FP16_TRAIN:", USE_FP16_TRAIN)

trainer_kwargs = dict(
    model=nllb_model,
    args=training_args,
    train_dataset=fit_tok,
    eval_dataset=dev_tok,
    data_collator=data_collator,
)

try:
    trainer = Seq2SeqTrainer(**trainer_kwargs, processing_class=nllb_tokenizer)
except TypeError:
    trainer = Seq2SeqTrainer(**trainer_kwargs, tokenizer=nllb_tokenizer)

prepare_nllb_lora_for_training(trainer.model, use_gradient_checkpointing=True)

_xb = next(iter(trainer.get_train_dataloader()))
assert (_xb["labels"] != -100).sum().item() > 0, "No valid label tokens."
_n_trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
assert _n_trainable > 0, "Trainer model has no trainable params"
print("trainer trainable param count:", _n_trainable)

# sanity: one backward step must reach LoRA weights
if torch.cuda.is_available():
    _san = {k: v.to(DEVICE) for k, v in _xb.items()}
    _out = trainer.model(**_san)
    trainer.model.zero_grad(set_to_none=True)
    _out.loss.backward()
    _has_grad = any(
        p.grad is not None and p.grad.abs().sum().item() > 0
        for p in trainer.model.parameters()
        if p.requires_grad
    )
    trainer.model.zero_grad(set_to_none=True)
    assert _has_grad, "LoRA backward produced no gradients — re-run LoRA cell from scratch"
    print("backward sanity: LoRA grads OK, loss=", float(_out.loss))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

trainer.train()

adapter_dir = WORK_DIR / "nllb_lora_sc_en_adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(adapter_dir)
nllb_tokenizer.save_pretrained(adapter_dir)
print("Saved LoRA adapter:", adapter_dir)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


USE_FP16_TRAIN: True
trainer trainable param count: 3538944


/tmp/ipykernel_57/1874822450.py:72: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("backward sanity: LoRA grads OK, loss=", float(_out.loss))


backward sanity: LoRA grads OK, loss= 9.375077247619629


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
50,7.966400
100,5.094500
150,3.590200
200,2.909300
250,2.433400
300,2.195800
350,2.042600
400,1.911900
450,1.930500
500,1.890100


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Saved LoRA adapter: /kaggle/working/nllb_lora_sc_en_adapter


## 5. Predict English on Test 200

Uses **`nllb_translate_lora_fp32`**: unwrap base → **`.float()`** on GPU, then **`generate(forced_bos_token_id=eng_Latn)`** only — matches §3 logic.


In [14]:
def generate_nllb_en_clean(texts, batch_size=EVAL_BATCH_SIZE):
    return nllb_translate_lora_fp32(nllb_model, nllb_tokenizer, texts, batch_size=batch_size)


_sanity_src = test_df[TEST_SC_COL].iloc[0]
_sanity_pred = generate_nllb_en_clean([_sanity_src])[0]
print("SC:", _sanity_src[:80])
print("Pred:", _sanity_pred[:220])

test_df[PRED_EN_COL] = generate_nllb_en_clean(test_df[TEST_SC_COL].tolist())
display(test_df[[TEST_SC_COL, TEST_EN_COL, PRED_EN_COL]].head(10))


SC: 我是捡镜子的命法莫法生了这个命哦命哦
Pred: I'm the one who gave birth to this life.


,text,english,pred_english_lora_nllb
0,我是捡镜子的命法莫法生了这个命哦命哦,I guess this is my fate—to pick up mirrors. Th...,I'm the one who gave birth to this life.
1,谢谢谢谢李婆婆,"Thank you, Grandma Li.","Thank you, Grandmother Lee."
2,我有几个朋友都强烈推荐他们家的小面说海椒香得很点一碗干的大款来搞哈调料就是这几种也不复杂没得...,Several of my friends strongly recommended the...,I have several friends who strongly recommend ...
3,完全由个人的爱好来决定你可以两样都做一次,It is entirely up to personal preference; you ...,It's entirely personal choice that you can do ...
4,真的还是碰对了伸手又把三筒摸回来这一下这个痛青就稳当了,That was the right Pong. I got the 3-dot right...,I really touch the right hand and touch the th...
5,给大家呢就是嗯普及一些知识就想说一下四川目前的民用机场那大家晓得哈四川航空呢其实是一个非常优...,Let me share some knowledge and talk about Sic...,I'm going to give you some knowledge about the...
6,那个时候大人忙农活和家务孩子呢不仅打不上手,"At that time, the adults were busy with farm w...","At that time, the adults were busy farming and..."
7,如果有买期房的朋友哪些房企的房产具有风险性我前一个视频发了但被和谐掉了在这里如果有需要知道哪...,"For friends who have bought presale homes, whi...","If you have friends who buy a house, which pro..."
8,这就有点无聊了要说保卫自己的领土激战一把哎倒也无所谓了赖在人家地盘死活都不走那个鬼子们还真的...,This is ridiculous. If they were defending the...,It's a bit boring to say that you're going to ...
9,啊就相当于民间的二人转的啊那么就写点这些啊比如说你要形容美女啊我以前都在说过哈长得来千娇百媚哈媚哈,This is similar to folk duet performance. For ...,"Oh, it's like two people turned around, so wri..."


## 6. Evaluate BLEU and chrF


In [15]:
def corpus_scores(preds, refs):
    preds = [str(x).strip() for x in preds]
    refs = [str(x).strip() for x in refs]
    return {
        "BLEU": sacrebleu.corpus_bleu(preds, [refs]).score,
        "chrF": sacrebleu.corpus_chrf(preds, [refs]).score,
    }


scores = corpus_scores(test_df[PRED_EN_COL].tolist(), test_df[TEST_EN_COL].tolist())
metrics_df = pd.DataFrame(
    [
        {
            "system": "S_tuned_LoRA_NLLB_SC_to_EN",
            "BLEU": scores["BLEU"],
            "chrF": scores["chrF"],
            "train_label": "NLLB pseudo English translated from train Mandarin",
            "test_reference": TEST_EN_COL,
        }
    ]
)
display(metrics_df)

metrics_path = WORK_DIR / "lora_nllb_sc_en_metrics.csv"
preds_save = test_df[[TEST_SC_COL, TEST_EN_COL, PRED_EN_COL]].rename(
    columns={TEST_SC_COL: "text", TEST_EN_COL: "english"}
)
preds_path = WORK_DIR / "lora_nllb_sc_en_predictions.csv"
metrics_df.to_csv(metrics_path, index=False)
preds_save.to_csv(preds_path, index=False, encoding="utf-8-sig")
print("Saved metrics:", metrics_path)
print("Saved predictions:", preds_path)


,system,BLEU,chrF,train_label,test_reference
0,S_tuned_LoRA_NLLB_SC_to_EN,6.021892,27.62822,NLLB pseudo English translated from train Mand...,english


Saved metrics: /kaggle/working/lora_nllb_sc_en_metrics.csv
Saved predictions: /kaggle/working/lora_nllb_sc_en_predictions.csv


## 7. Qualitative Cases for Address Terms


In [16]:
KEY_TERMS = [
    "老汉",
    "嬢嬢",
    "幺儿",
    "幺妹",
    "婆婆",
    "老表",
    "堂客",
    "自家",
    "别个",
    "人家",
    "老师",
    "老哥",
    "美女",
    "帅哥",
    "伙计",
    "老乡",
    "老子",
    "雄起",
]

mask = test_df[TEST_SC_COL].astype(str).apply(lambda s: any(term in s for term in KEY_TERMS))
cases = test_df.loc[mask, [TEST_SC_COL, TEST_EN_COL, PRED_EN_COL]].copy()
if len(cases) == 0:
    cases = test_df[[TEST_SC_COL, TEST_EN_COL, PRED_EN_COL]].copy()

cases = cases.head(50)
cases["sentence_BLEU"] = [
    sacrebleu.sentence_bleu(pred, [ref]).score
    for pred, ref in zip(cases[PRED_EN_COL].astype(str), cases[TEST_EN_COL].astype(str))
]

cases_path = WORK_DIR / "lora_nllb_sc_en_cases.csv"
cases.to_csv(cases_path, index=False, encoding="utf-8-sig")
display(cases.head(20))
print("Saved cases:", cases_path)


,text,english,pred_english_lora_nllb,sentence_BLEU
1,谢谢谢谢李婆婆,"Thank you, Grandma Li.","Thank you, Grandmother Lee.",32.466792
8,这就有点无聊了要说保卫自己的领土激战一把哎倒也无所谓了赖在人家地盘死活都不走那个鬼子们还真的...,This is ridiculous. If they were defending the...,It's a bit boring to say that you're going to ...,2.403073
9,啊就相当于民间的二人转的啊那么就写点这些啊比如说你要形容美女啊我以前都在说过哈长得来千娇百媚哈媚哈,This is similar to folk duet performance. For ...,"Oh, it's like two people turned around, so wri...",7.639864
49,有天我无意翻到了格子心老师的文章写的太好了至少我写不了自愧不如他应该的确是有格子心的只是一直...,One day I happened to come across an article b...,One day I didn't want to turn over to the teac...,2.075776
72,还是搞成了好几对的哈因为你是个外地的是个外地所以我们来今天问一下嘛美女你们和那个电话老板聊得...,It still ended up forming several pairs. Since...,"It's like a couple of couples, because you're ...",3.078746
121,甜吃一口嘛别的婆娘米奇我呀黑珍珠的育儿激情而且我这个是专门买起来的哦我我不是说什么爱你专门买...,Have a bite of something sweet. The other wome...,I'm not saying we're going to have some sweet ...,0.454803
154,你既应了就得耐烦些少不得替人家办办,"Since you have agreed, you need to be more pat...","You're supposed to be a little patient, and yo...",8.634081
185,他们十几年的施工团队专业且负责材料家具家电方面更不用说超大独栋大楼整整三层全是工厂旗舰店儿厂...,They have a construction team with more than t...,"They're a decade old construction team, and th...",2.284872


Saved cases: /kaggle/working/lora_nllb_sc_en_cases.csv


## 8. Report Wording
#
Model name:
- `S_tuned: LoRA-NLLB end-to-end Sichuanese -> English`
#
Training data:
- Source: `train_9800_zh.csv` Sichuanese `text`
- Target: pseudo English (`pseudo_english_nllb`) generated by vanilla NLLB from the Mandarin pivot column
#
Evaluation:
- Test: `golden_manually_200_en.csv`
- Reference: human/gold English column
- Metrics: BLEU and chrF (SacreBLEU)
#
Note:
- Because the 9800 training rows do not have human English, the training
  target is pseudo-labeled English. This should be stated clearly in the
  report.
